In [1]:
import os
import tarfile
import json
import io
from pathlib import Path

def create_clip_webdataset(root_dir, output_filename):
    root_path = Path(root_dir)
    # Get all ingredient folders (ignoring hidden files)
    ingredient_folders = [f for f in root_path.iterdir() if f.is_dir()]
    
    with tarfile.open(output_filename, "w") as tar:
        count = 0
        for folder in ingredient_folders:
            ingredient_name = folder.name
            # Path to the thumbs directory
            thumbs_path = folder / "thumbs"
            
            if not thumbs_path.exists():
                continue
                
            for img_path in thumbs_path.glob("*"):
                if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.webp']:
                    continue
                
                # Create a unique basename for this sample
                basename = f"{count:06d}"
                
                # 1. Add Image File
                tar.add(img_path, arcname=f"{basename}{img_path.suffix}")
                
                # 2. Add Metadata (JSON) for Prompt Ensembling
                # We store the raw ingredient and a few pre-generated prompts
                metadata = {
                    "ingredient": ingredient_name,
                    "prompts": [
                        f"a photo of {ingredient_name}",
                        f"a close-up shot of {ingredient_name}",
                        f"fresh {ingredient_name} for cooking",
                        f"an image of the ingredient {ingredient_name}",
                        f"a photo of raw {ingredient_name}",
                        f"{ingredient_name} on a white background",
                        f"fresh {ingredient_name} on a cutting board",
                        f"{ingredient_name} at a grocery store",
                        f"a high quality photo of {ingredient_name}"
                    ]
                }
                json_data = json.dumps(metadata).encode('utf-8')
                json_info = tarfile.TarInfo(name=f"{basename}.json")
                json_info.size = len(json_data)
                tar.addfile(json_info, io.BytesIO(json_data))
                
                count += 1
                if count % 100 == 0:
                    print(f"Packed {count} images...")

# Usage
create_clip_webdataset("./saved_pages", "ingredients_data.tar")

Packed 100 images...


## Using .tar file with CLIP and Prompt Ensembling

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-2lhl23vm
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-2lhl23vm
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.2 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=f406bc318125ed0a6bca183d23261d333553e572ca2edfc9c37aa420a7941142
  Stored in directory: /tmp/pip-ephem-wheel-cache-5bde2bsv/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [7]:
pip install ftfy

In [8]:
pip install webdataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 4.7 MB/s eta 0:00:00


In [ ]:
import torch
import clip
import webdataset as wds

# 1. Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

# ==========================================
# THE FIX: Apply preprocess in the pipeline
# ==========================================
dataset = (
    wds.WebDataset("/content/drive/My Drive/Colab Notebooks/ingredients_data.tar")
    .decode("pil") 
    .to_tuple("jpg;png;webp", "json") 
    # Map 'preprocess' to the 1st item (image), and a dummy lambda to the 2nd item (metadata)
    .map_tuple(preprocess, lambda meta: meta) 
)

# Now it works! The DataLoader receives [3, 224, 224] Tensors and can stack them.
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)

# ==========================================
# UPDATE: Accept 'prompts' directly
# ==========================================
def ensemble_inference(image_tensor, prompts):
    image_input = image_tensor.unsqueeze(0).to(device)
    
    # We pass the extracted prompts list directly here
    text_inputs = clip.tokenize(prompts).to(device)
    
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_inputs)
        
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        mean_text_feature = text_features.mean(dim=0, keepdim=True)
        mean_text_feature /= mean_text_feature.norm(dim=-1, keepdim=True)
        
        similarity = (100.0 * image_features @ mean_text_feature.T)
        
    return similarity

# ==========================================
# THE FIX: Extracting from Dictionary of Lists
# ==========================================
for batched_imgs, batched_meta in dataloader:
    
    # 1. Get the ingredient name for the first image in the batch
    ingredient_0 = batched_meta['ingredient'][12]
    
    # 2. Extract the prompts for the first image. 
    # Because PyTorch transposed the lists, we use a quick list comprehension 
    # to grab the 0th item from each transposed tuple.
    prompts_0 = [prompt_tuple[12] for prompt_tuple in batched_meta['prompts']]
    
    # 3. Run the inference
    score = ensemble_inference(batched_imgs[12], prompts_0)
    
    print(f"Confidence score for {ingredient_0}: {score.item():.2f}")
    break

/usr/local/lib/python3.12/dist-packages/webdataset/compat.py:379: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn("WebDataset(shardshuffle=...) is None; set explicitly to False or a number")


Confidence score for abalone: 38.25


In [23]:
import pandas as pd

df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/canonical_ingredients.csv")

In [ ]:
# A list of all possible ingredients in your app
all_categories = list(df["canonical_ingredient"].unique()) # add all 12

def predict_ingredient(image_tensor):
    image_input = image_tensor.unsqueeze(0).to(device)
    
    # Create a basic prompt for EVERY category
    text_prompts = [f"a photo of {cat}" for cat in all_categories]
    text_inputs = clip.tokenize(text_prompts).to(device)
    
    with torch.no_grad():
        image_features = model.encode_image(image_input)
        text_features = model.encode_text(text_inputs)
        
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)
        
        # Calculate similarity against ALL prompts at once
        similarity = (100.0 * image_features @ text_features.T)
        
        # Use Softmax to turn the raw scores into percentages that add up to 100%
        probabilities = similarity.softmax(dim=-1).cpu().numpy()[0]
    
    # Find the index of the highest probability
    best_index = probabilities.argmax()
    predicted_ingredient = all_categories[best_index]
    confidence = probabilities[best_index] * 100
    
    return predicted_ingredient, confidence

# Test it in your loop
for batched_imgs, batched_meta in dataloader:
    val = random.randint(0, len)  # Randomly select an index in the batch
    actual_ingredient = batched_meta['ingredient'][0]
    prediction, confidence = predict_ingredient(batched_imgs[0])
    
    print(f"Actual: {actual_ingredient}")
    print(f"Predicted: {prediction} (Confidence: {confidence:.1f}%)")
    break

Actual: abalone
Predicted: abalone (Confidence: 27.2%)
